# KG1 v74 — DARE-TIES Adapter Merge + LoRAhub CMA-ES (Colab A100)

## 3-way merge: huikang_v26 + samvalladares + V73-GRPO

**Bombas**:
- DARE-TIES: drop+rescale para sparsificar antes de merge
- LoRAhub CMA-ES: black-box optimization de pesos
- Validação local em 950 problems antes de submit

## Score esperado: 0.87 → 0.88 (P=50-60%)

In [ ]:
# Cell 1: Setup
import torch, os, subprocess
subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True)
%pip install -q peft>=0.18.1 transformers>=4.55 accelerate bitsandbytes nevergrad
%pip install -q huggingface_hub safetensors kaggle
from google.colab import drive, userdata
drive.mount('/content/drive')

try:
    HF_TOKEN = userdata.get('HF_KEY')
except Exception:
    HF_TOKEN = userdata.get('HF_TOKEN', '')
assert HF_TOKEN.startswith('hf_'), 'Configure HF_KEY no Colab Secrets'
os.environ['HF_TOKEN'] = HF_TOKEN


In [ ]:
# Cell 2: Download 3 adapters (Drive + HF + Kaggle)
from huggingface_hub import snapshot_download
import os, shutil, json as _json

# Adapter 1: V73-GRPO (nosso final)
V73_GRPO = '/content/drive/MyDrive/kg1_v73_grpo/final_grpo'
if not os.path.exists(V73_GRPO):
    try:
        V73_GRPO = snapshot_download('felipesp1983/kg1-nemotron-lora-v73-grpo',
                                      token=HF_TOKEN, allow_patterns=['final/*']) + '/final'
    except Exception as e:
        print(f'WARN: V73-GRPO download falhou ({e}), usando V73-SFT')
        V73_GRPO = snapshot_download('felipesp1983/kg1-nemotron-lora-v73-unsloth-moe',
                                      token=HF_TOKEN, allow_patterns=['final/*']) + '/final'

# Adapter 2: huikang v26 via Kaggle (samvalladares dataset)
HUIKANG_V26 = '/content/huikang_v26'
os.makedirs(HUIKANG_V26, exist_ok=True)

# Configure Kaggle credentials (try Drive first, fallback to userdata)
os.makedirs('/root/.kaggle', exist_ok=True)
kaggle_json_drive = '/content/drive/MyDrive/.kaggle/kaggle.json'
if os.path.exists(kaggle_json_drive):
    shutil.copy(kaggle_json_drive, '/root/.kaggle/kaggle.json')
else:
    try:
        kaggle_user = userdata.get('KAGGLE_USERNAME')
        kaggle_key = userdata.get('KAGGLE_KEY')
        with open('/root/.kaggle/kaggle.json', 'w') as f:
            _json.dump({'username': kaggle_user, 'key': kaggle_key}, f)
    except Exception as e:
        print(f'ERRO: Configure kaggle.json no Drive ou KAGGLE_USERNAME/KAGGLE_KEY nos Secrets: {e}')
        raise
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Download huikang v26 artifacts via os.system (parseable)
_has_weights = any(f.endswith('.safetensors') for f in os.listdir(HUIKANG_V26))
if not _has_weights:
    rc = os.system(f'kaggle datasets download -d samvalladares/huikang-nemotron-artifacts -p {HUIKANG_V26} --unzip')
    if rc != 0:
        print('ERRO: kaggle download falhou')
        raise RuntimeError('Kaggle download failed')

print(f'Adapters downloaded:')
print(f'  V73_GRPO: {V73_GRPO}')
print(f'  HUIKANG_V26: {HUIKANG_V26}')
print(f'    files: {os.listdir(HUIKANG_V26)[:5]}')


In [ ]:
# Cell 3: Load model + 3 adapters via PEFT
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16',
    quantization_config=bnb, device_map='auto', trust_remote_code=True, token=HF_TOKEN,
)
tok = AutoTokenizer.from_pretrained('nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16', trust_remote_code=True, token=HF_TOKEN)

# Load 2 adapters (V73-GRPO + huikang v26)
model = PeftModel.from_pretrained(model, V73_GRPO, adapter_name='v73')
model.load_adapter(HUIKANG_V26, adapter_name='huikang_v26')
print('Adapters loaded:', list(model.peft_config.keys()))

In [ ]:
# Cell 4: DARE-TIES merge com weights iniciais
WEIGHTS_INIT = [0.6, 0.4]  # v73 dominante, huikang complementar

model.add_weighted_adapter(
    adapters=['v73', 'huikang_v26'],
    weights=WEIGHTS_INIT,
    adapter_name='dare_ties_v1',
    combination_type='dare_ties',
    density=0.7,            # drop 30%
)
model.set_adapter('dare_ties_v1')
print('DARE-TIES merge v1 created')

In [ ]:
# Cell 5: Local pre-score em validation set (OPCIONAL - precisa cleanup memory)
%pip install -q vllm>=0.6
import torch, gc

# CRITICO: PEFT model na GPU + vLLM = OOM em A100 40GB (vLLM carrega BF16 = 60GB)
# Solucao 1: salvar adapter, deletar model, carregar vLLM novo
# Solucao 2: skip esta celula e usar V74_INFERENCE_KAGGLE.ipynb (separado)

# Save merged adapter to disk for vLLM loading (PEFT 0.18+ usa adapter ativo)
MERGED_DIR = '/content/merged_adapter'
model.save_pretrained(MERGED_DIR)
print(f'Merged adapter salvo em {MERGED_DIR}: {os.listdir(MERGED_DIR)[:5]}')

# IMPORTANTE: liberar memory antes de carregar vLLM
print(f'GPU mem antes cleanup: {torch.cuda.memory_allocated()/1e9:.1f}GB')
del model
gc.collect()
torch.cuda.empty_cache()
print(f'GPU mem apos cleanup: {torch.cuda.memory_allocated()/1e9:.1f}GB')

# OPCIONAL: pre-score (skip se quer apenas criar adapter)
DO_PRESCORE = False  # Mude para True se quer validar score local antes de upload
if DO_PRESCORE:
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    import pandas as pd

    # Validation set (subset 100 prompts)
    if os.path.exists('/content/drive/MyDrive/kg1_train.csv'):
        df = pd.read_csv('/content/drive/MyDrive/kg1_train.csv')
    else:
        rc = os.system('kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -f train.csv -p /content/')
        if rc != 0:
            print('WARN: kaggle download falhou, pulando prescore')
            DO_PRESCORE = False

    if DO_PRESCORE:
        df = pd.read_csv('/content/train.csv')
        val_problems = list(df['prompt'].head(100))
        llm = LLM(model='nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16',
                  enable_lora=True, max_lora_rank=32,
                  gpu_memory_utilization=0.85, dtype='bfloat16',
                  max_model_len=8192)
        sp = SamplingParams(temperature=0.0, max_tokens=4096)
        lora_req = LoRARequest('merged', 1, MERGED_DIR)
        outs = llm.generate(val_problems, sp, lora_request=lora_req)
        print(f'Generated {len(outs)} outputs')

        # IMPORTANTE: cleanup vLLM antes de continuar para Cell 6/7
        del llm
        gc.collect()
        torch.cuda.empty_cache()
        print('vLLM destroyed, ready for Cell 6/7')
else:
    print('Prescore skipped. MERGED_DIR pronto para upload via Cell 7')


In [ ]:
# Cell 6: LoRAhub CMA-ES otimizacao (DOCUMENTACAO - implementacao requer eval loop)
# IMPORTANTE: Cell 5 ja deletou 'model' para evitar OOM com vLLM.
# Para implementar CMA-ES de verdade:
# 1. Re-carregar PEFT model (Cell 3 logic)
# 2. Re-load 2 adapters (v73 + huikang_v26)
# 3. Loop optimizer.minimize com weights diferentes
# 4. Para cada iteracao: re-merge + eval em val_set + return -accuracy
#
# Para FASE 4 SIMPLES, usamos weights iniciais [0.6, 0.4] proven (do roadmap):

import nevergrad as ng

best_weights = [0.6, 0.4]  # v73_GRPO (peso maior) + huikang_v26 (complementar)
print(f'Using weights: {best_weights}')
print('CMA-ES skipped (placeholder). Para v75: implementar full CMA-ES com val_set proper.')


In [ ]:
# Cell 7: Upload V74 final para HF
# MERGED_DIR foi criado em Cell 5 (model.save_pretrained antes de del model)
from huggingface_hub import HfApi

if not os.path.exists(MERGED_DIR):
    raise RuntimeError(f'{MERGED_DIR} nao existe. Execute Cell 5 primeiro (sem skip).')

print(f'V74 adapter em {MERGED_DIR}: {os.listdir(MERGED_DIR)}')

# Verifica se tem os 2 arquivos obrigatorios para Kaggle
required = ['adapter_config.json', 'adapter_model.safetensors']
present = os.listdir(MERGED_DIR)
missing = [f for f in required if f not in present]
if missing:
    print(f'WARN: faltando arquivos para Kaggle submit: {missing}')
    print('Verifique se Cell 4 (DARE-TIES merge) executou corretamente')
else:
    print('OK: adapter_config.json + adapter_model.safetensors presentes')

# Upload to HF
api = HfApi(token=HF_TOKEN)
REPO_ID = 'felipesp1983/kg1-nemotron-lora-v74-dare-ties'
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(folder_path=MERGED_DIR, repo_id=REPO_ID, path_in_repo='final')
print(f'V74 uploaded to {REPO_ID}')
print(f'Para Kaggle: download esses 2 files, zip como submission.zip, kaggle competitions submit')
